# 实验10：CANN 工具进行 AI 模型性能分析

## 一、实验目的

本实验基于CANNLab一站式开发平台或配置有CANN Toolkit的云环境，使同学们掌握使用CANN（Compute Architecture for Neural Networks）性能分析及可视化工具进行AI模型训练与推理性能分析的方法。通过本实验，同学们将：

1. 了解CANN性能分析工具链的整体架构与核心能力；
2. 掌握msprof命令行工具进行性能数据采集与解析的方法；
3. 学会使用Ascend PyTorch Profiler和MindSpore Profiler接口进行框架级性能分析；
4. 掌握MindStudio Insight工具进行性能数据可视化展示的能力；
5. 具备识别AI模型常见性能瓶颈（Kernel Launch开销、内存带宽受限、数据传输瓶颈等）并定位根因的能力；
6. 了解典型性能优化策略与专家系统辅助分析工具的使用方法。

## 二、实验说明

### 2.1 实验背景

在AI模型开发与部署过程中，“模型能够跑起来”只是第一步，“跑得快、跑得稳、跑得省”才是工程落地的核心目标。随着模型规模不断增长（如大语言模型、多模态模型等），算力需求呈指数级上升，传统CPU计算模式已难以满足需求，异构计算通过整合CPU、GPU、NPU等不同架构的处理器形成协同计算网络，成为突破算力瓶颈的关键路径。昇腾AI处理器（NPU）凭借其强大的计算能力为AI应用提供了高性能的硬件基础，然而许多开发者在使用NPU时仅停留在“调用API”的层面，未能充分发挥硬件潜力。

CANN作为昇腾AI处理器的全栈软件平台，不仅提供了高性能算子库和图编译器，更内置了完整的性能分析与调优工具链。CANN性能分析工具链提供了三层性能观测能力：应用层Profiling（Python API）、运行时层Trace（C++/JSON）和硬件计数器（Hardware Counters），覆盖从顶层应用到底层硬件的全栈性能数据采集与分析。


### 2.2 实验环境（**CANNLab在线实验跳过**）

- **硬件环境**：昇腾AI处理器（Atlas 300T/800T或Atlas 200DK等），本文以NPU 910B环境为例
- **软件环境**：
  - 操作系统：EulerOS / Ubuntu 18.04+
  - CANN版本：8.0及以上（推荐8.0或更高版本）
  - AI框架：PyTorch（torch_npu）/ MindSpore 2.3.0+
  - Python版本：3.8+
- **工具清单**：
  - msprof（命令行性能采集工具）
  - msprof-analyze（性能数据统计分析工具）
  - Ascend PyTorch Profiler / MindSpore Profiler（框架级性能采集接口）
  - MindStudio Insight（性能数据可视化分析工具）
  - msadvisor（专家系统分析工具）

## 三、实验任务

### 3.1 任务描述

本实验分为三个核心任务：

**任务一**：使用msprof命令行工具采集AI模型推理的性能数据，并解析关键性能指标文件。

**任务二**：使用框架Profiler接口（Ascend PyTorch Profiler或MindSpore Profiler）采集训练过程中的性能数据，并使用MindStudio Insight工具进行可视化分析。

**任务三**：基于采集的性能数据，识别模型性能瓶颈，并结合优化策略完成一轮性能优化验证。

### 3.2 学习目标

完成本实验后，同学们应能够：

- 独立搭建CANN性能分析环境，正确配置Profiling参数
- 熟练使用msprof命令行工具完成性能数据采集
- 理解op_summary.csv、task_time.csv、api_statistic.csv等核心输出文件的含义
- 使用MindStudio Insight工具进行Timeline视图分析
- 识别常见的性能瓶颈类型（Kernel Launch开销、内存带宽受限、数据搬运瓶颈、AI CPU算子瓶颈等）
- 应用至少一种性能优化策略（算子融合、数据布局调整、多流并行等）并验证优化效果

## 四、任务准备

### 4.1 前置知识

1. **CANN架构基础**：了解CANN的异构计算架构，包括图编译器、算子库、运行时系统等核心组件的作用。
2. **昇腾AI处理器基础**：了解昇腾NPU的基本架构，包括AI Core（计算核心）、AI CPU（通用计算单元）、L2缓存等硬件单元的功能。
3. **性能分析基本概念**：
   - **Kernel Launch开销**：每次启动Kernel（算子执行单元）都需要一定的调度开销，大量小算子会导致频繁的Kernel启动，造成性能损失
   - **内存带宽受限**：当数据搬运速度成为瓶颈时，即使计算单元处理能力强也无法发挥全部性能
   - **流水线并行度**：计算单元与数据搬运单元的重叠执行程度
   - **Host-Device数据传输**：CPU与NPU之间的数据拷贝耗时
4. **PyTorch/MindSpore基础**：具备基本的模型训练与推理脚本编写能力。

### 4.2 环境检查（**CANNLab在线实验跳过**）

在开始实验前，请依次完成以下环境检查：

**Step 1：检查NPU硬件状态**

```bash
npu-smi info
```

该命令用于查询昇腾NPU硬件的基本信息，包括芯片型号、内存容量、温度、功耗和使用率等。如果命令回显正常显示NPU设备信息，说明硬件状态正常。

**Step 2：检查CANN环境变量**

```bash
# 加载CANN环境变量（根据实际安装路径调整）
source /usr/local/Ascend/ascend-toolkit/set_env.sh
```

该脚本用于设置CANN工具链所需的所有环境变量，包括可执行文件路径、库文件路径和Python包路径等。每次新开终端都需要执行此命令。

**Step 3：检查AI框架版本**

```bash
pip list | grep torch
pip list | grep torch-npu
# 或者
pip list | grep mindspore
```

确认PyTorch（含torch_npu插件）或MindSpore框架已正确安装。

**Step 4：检查Profiling工具可用性**

```bash
# 进入CANN Profiler工具目录
ls /usr/local/Ascend/ascend-toolkit/latest/tools/profiler/bin/
```

确认msprof可执行文件存在。


### 4.3 实验数据准备

**准备测试模型**：可以选择以下任一方式准备实验模型：

- 使用PyTorch官方预训练模型（如ResNet-50、MobileNetV2等），通过torch_npu迁移至NPU
- 使用MindSpore Model Zoo中的预训练模型
- 自行构建简单测试模型（如多层卷积网络、MLP模型等）

**准备测试数据**：使用随机生成的张量数据即可满足性能分析需求。

## 五、任务实施

### 5.1 任务一：使用msprof命令行工具采集推理性能数据

#### 5.1.1 理解msprof工具

msprof是CANN工具链中最基础也最强大的命令行性能分析工具，它直接与昇腾AI处理器硬件计数器交互，提供最原始的性能数据。msprof命令行工具提供了AI任务运行性能数据、昇腾AI处理器系统数据等性能数据的采集和解析能力。


#### 5.1.2 编写测试模型脚本

首先创建一个简单的PyTorch模型脚本用于性能分析。该脚本定义一个包含卷积、批归一化和全连接层的简单网络，执行若干次推理迭代。

```python
# test_model.py
import torch
import torch.nn as nn
import torch_npu
import time

# 指定NPU设备
torch.npu.set_device(0)

# 定义一个简单的CNN模型
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, 10)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# 主执行逻辑
if __name__ == "__main__":
    model = SimpleModel().to("npu")
    model.eval()
    
    # 构造随机输入数据
    input_data = torch.randn(8, 3, 224, 224).to("npu")
    
    # 预热（Warm-Up）：前几次推理包含模型初始化开销，需排除以获取稳态性能
    print("Warming up...")
    for _ in range(5):
        with torch.no_grad():
            _ = model(input_data)
    torch.npu.synchronize()
    
    # 正式推理
    print("Running inference...")
    start = time.time()
    for _ in range(100):
        with torch.no_grad():
            _ = model(input_data)
    torch.npu.synchronize()
    end = time.time()
    
    print(f"Average inference time: {(end - start) / 100 * 1000:.2f} ms")
```


**代码说明**：
- `torch.npu.set_device(0)`：指定使用第0号NPU设备
- `model.to("npu")`：将模型参数和数据移至NPU
- `model.eval()`：将模型设置为评估模式，关闭Dropout等训练特性
- 预热阶段（Warm-Up）：前5次推理用于触发模型编译和算子初始化，排除首次执行的特殊开销
- `torch.npu.synchronize()`：同步等待所有NPU异步操作完成，确保计时准确

In [2]:
import torch
import torch.nn as nn
import torch_npu
import time

# 指定NPU设备
torch.npu.set_device(0)

# 定义一个简单的CNN模型
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, 10)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        
        return x
        

In [3]:
model = SimpleModel().to("npu")
model.eval()

# 构造随机输入数据
input_data = torch.randn(8, 3, 224, 224).to("npu")

# 预热（Warm-Up）：前几次推理包含模型初始化开销，需排除以获取稳态性能
print("Warming up...")
for _ in range(5):
    with torch.no_grad():
        _ = model(input_data)
torch.npu.synchronize()

# 正式推理
print("Running inference...")
start = time.time()
for _ in range(100):
    with torch.no_grad():
        _ = model(input_data)
torch.npu.synchronize()
end = time.time()

print(f"Average inference time: {(end - start) / 100 * 1000:.2f} ms")

Warming up...
Running inference...
Average inference time: 1.24 ms


#### 5.1.3 使用msprof采集性能数据

使用msprof命令行工具采集上述脚本的性能数据：

```bash
# 进入Profiler工具目录
cd /usr/local/Ascend/ascend-toolkit/latest/tools/profiler/bin

# 使用msprof采集性能数据
msprof --output=./profiling_output \
       --model-execution=on \
       --runtime-api=on \
       --aicpu=on \
       --ai-core=on \
       --sys-hardware-mem=on \
       python3 test_model.py
```

**参数说明**：
- `--output`：指定性能数据的输出存放路径
- `--model-execution=on`：采集模型执行数据
- `--runtime-api=on`：采集Runtime API调用数据
- `--aicpu=on`：采集AI CPU性能数据
- `--ai-core=on`：采集AI Core性能数据（关键参数）
- `--sys-hardware-mem=on`：采集L2缓存命中率数据

**采集完成标志**：命令执行后若回显中包含“Profiling finished”和“Process profiling data complete”信息，表示采集完成。

#### 5.1.4 查看采集结果

采集完成后，在`--output`指定的目录下会生成`PROF_XXX`目录，其中包含以下关键文件：

| 文件名 | 说明 |
|--------|------|
| `msprof_*.json` | Timeline数据总表，包含所有事件的时间线信息 |
| `op_summary_*.csv` | AI Core和AI CPU算子详细数据 |
| `op_statistic_*.csv` | AI Core和AI CPU算子调用次数及耗时统计 |
| `task_time_*.csv` | Task Scheduler任务调度信息 |
| `api_statistic_*.csv` | CANN层API执行耗时信息统计 |
| `README.txt` | 采集信息说明文件 |

**查看操作耗时统计**：

```bash
# 查看算子耗时统计
cat profiling_output/PROF_XXX/mindstudio_profiler_output/op_statistic_*.csv | head -20
```

该CSV文件列出了每个算子的总耗时、平均耗时、调用次数等信息，按总耗时降序排列，最前面的算子即为当前模型的计算热点。

**查看API耗时统计**：

```bash
cat profiling_output/PROF_XXX/mindstudio_profiler_output/api_statistic_*.csv | head -20
```

API耗时统计反映了CANN层（包括AscendCL、Runtime等）各API的执行耗时情况。


### 5.2 任务二：使用框架Profiler接口采集训练性能数据并可视化

#### 5.2.1 使用Ascend PyTorch Profiler

Ascend PyTorch Profiler是CANN针对PyTorch框架开发的性能分析工具，通过在PyTorch脚本中添加Profiler接口，在模型执行的同时采集性能数据，执行完成后直接输出可视化的性能数据文件。

创建训练性能分析脚本：

```python
# train_profiling.py
import torch
import torch.nn as nn
import torch.optim as optim
import torch_npu
from torch_npu.profiler import profile, ProfilerActivity, tensorboard_trace_handler

# 指定NPU设备
torch.npu.set_device(0)

# 定义训练模型
class TrainModel(nn.Module):
    def __init__(self):
        super(TrainModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.fc1 = nn.Linear(128 * 56 * 56, 512)
        self.fc2 = nn.Linear(512, 10)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.maxpool(x)
        x = self.relu(self.conv2(x))
        x = self.maxpool(x)
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

if __name__ == "__main__":
    model = TrainModel().to("npu")
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    
    # 模拟训练数据
    inputs = torch.randn(16, 3, 224, 224).to("npu")
    labels = torch.randint(0, 10, (16,)).to("npu")
    
    # 配置Profiler采集训练过程性能数据
    # activities: 指定采集的Activity类型（NPU和CPU）
    # schedule: 配置采集策略，wait=2表示跳过前2步，warmup=2表示预热2步，
    #           active=3表示实际采集3步，repeat=1表示重复1轮
    # on_trace_ready: 指定性能数据输出方式，此处使用TensorBoard格式
    with profile(
        activities=[ProfilerActivity.NPU, ProfilerActivity.CPU],
        schedule=torch.profiler.schedule(wait=2, warmup=2, active=3, repeat=1),
        on_trace_ready=tensorboard_trace_handler("./profiler_logs"),
        record_shapes=True,
        profile_memory=True,
        with_stack=True
    ) as prof:
        for step in range(10):
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            torch.npu.synchronize()
            prof.step()
            print(f"Step {step}, Loss: {loss.item():.4f}")
    
    # 打印Profiler关键指标表格
    print(prof.key_averages().table(sort_by="npu_time_total", row_limit=15))
```

**核心参数说明**：
- `activities`：指定采集NPU和CPU端的性能事件
- `schedule`：采集调度策略，通过wait/warmup/active/repeat参数控制采集窗口，避免采集初始化阶段的不稳定数据
- `tensorboard_trace_handler`：将数据输出为TensorBoard可读格式
- `record_shapes=True`：记录张量形状信息
- `profile_memory=True`：记录NPU显存使用情况
- `with_stack=True`：记录调用栈，便于追溯耗时操作的来源


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_npu
from torch_npu.profiler import profile, ProfilerActivity, tensorboard_trace_handler

# 指定NPU设备
torch.npu.set_device(0)

# 定义训练模型
class TrainModel(nn.Module):
    def __init__(self):
        super(TrainModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.fc1 = nn.Linear(128 * 56 * 56, 512)
        self.fc2 = nn.Linear(512, 10)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.maxpool(x)
        x = self.relu(self.conv2(x))
        x = self.maxpool(x)
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [5]:
model = TrainModel().to("npu")
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# 模拟训练数据
inputs = torch.randn(16, 3, 224, 224).to("npu")
labels = torch.randint(0, 10, (16,)).to("npu")


In [6]:
# 配置Profiler采集训练过程性能数据
# activities: 指定采集的Activity类型（NPU和CPU）
# schedule: 配置采集策略，wait=2表示跳过前2步，warmup=2表示预热2步，
#           active=3表示实际采集3步，repeat=1表示重复1轮
# on_trace_ready: 指定性能数据输出方式，此处使用TensorBoard格式
with profile(
    activities=[ProfilerActivity.NPU, ProfilerActivity.CPU],
    schedule=torch.profiler.schedule(wait=2, warmup=2, active=3, repeat=1),
    on_trace_ready=tensorboard_trace_handler("./profiler_logs"),
    record_shapes=True,
    profile_memory=True,
    with_stack=True
) as prof:
    for step in range(10):
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        torch.npu.synchronize()
        prof.step()
        print(f"Step {step}, Loss: {loss.item():.4f}")

# 打印Profiler关键指标表格
print(prof.key_averages().table(sort_by="npu_time_total", row_limit=15))

[2026-08-04 09:17:54] [WARNING] [1922] profiler.py: Please use level1 or level2 if you want to collect aic metrics, reset aic metrics to None!
[2026-08-04 09:17:54] [WARNING] [1922] profiler.py: Please use level1 or level2 if you want to collect aic metrics, reset aic metrics to None!


/home/developer/.local/lib/python3.11/site-packages/torch/autograd/__init__.py:221: UserWarning: Cannot create tensor with interal format while allow_internel_format=False, tensor will be created with base format. (Triggered internally at ../torch_npu/csrc/aten/common/TensorFactories.cpp:340.)
  torch.ones_like(out, memory_format=torch.preserve_format)


Step 0, Loss: 2.3022
Step 1, Loss: 2.1799
Step 2, Loss: 2.0798
Step 3, Loss: 2.0115
Step 4, Loss: 1.9687
Step 5, Loss: 2.2733
Step 6, Loss: 2.2123
Step 7, Loss: 1.6514
Step 8, Loss: 1.5201
Step 9, Loss: 1.7042


AttributeError: 'profile' object has no attribute 'key_averages'

#### 5.2.2 使用MindSpore Profiler（MindSpore场景）

如果实验环境使用MindSpore框架，可以使用MindSpore Profiler接口进行性能分析。

```python
# mindspore_profiling.py
import mindspore
import mindspore.nn as nn
import mindspore.ops as ops
from mindspore import Profiler, Tensor
import numpy as np

mindspore.set_context(device_target="Ascend")

# 启用Profiler，输出路径为./data/profiler
profiler = Profiler(output_path="./data/profiler")

class SimpleNet(nn.Cell):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, pad_mode="pad", padding=1)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc = nn.Dense(64 * 112 * 112, 10)
    
    def construct(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

net = SimpleNet()
x = Tensor(np.random.randn(8, 3, 224, 224).astype(np.float32))

# 执行模型
for _ in range(10):
    output = net(x)

# 结束Profiling并导出数据
profiler.analyse()
```

In [2]:
import mindspore
import mindspore.nn as nn
import mindspore.ops as ops
from mindspore import Profiler, Tensor
import numpy as np

mindspore.set_context(device_target="Ascend")

# 启用Profiler，输出路径为./data/profiler
profiler = Profiler(output_path="/home/developer/experiment/lab_10/data/profiler")

class SimpleNet(nn.Cell):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, pad_mode="pad", padding=1)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc = nn.Dense(64 * 112 * 112, 10)
    
    def construct(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

[WARNING] ME(1846:281473372323856,MainProcess):2026-08-04-10:45:40.978.000 [mindspore/run_check/_check_version.py:344] MindSpore version 2.9.0 and "te" wheel package version 7.8 does not match. For details, refer to the installation guidelines: https://www.mindspore.cn/install
[WARNING] ME(1846:281473372323856,MainProcess):2026-08-04-10:45:40.978.000 [mindspore/run_check/_check_version.py:357] Please pay attention to the above warning, countdown: 3
[WARNING] ME(1846:281473372323856,MainProcess):2026-08-04-10:45:41.979.000 [mindspore/run_check/_check_version.py:357] Please pay attention to the above warning, countdown: 2
[WARNING] ME(1846:281473372323856,MainProcess):2026-08-04-10:45:42.980.000 [mindspore/run_check/_check_version.py:357] Please pay attention to the above warning, countdown: 1
[WARNING] ME(1846:281473372323856,MainProcess):2026-08-04-10:45:43.982.000 [mindspore/context.py:1338] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a f

In [3]:
net = SimpleNet()
x = Tensor(np.random.randn(8, 3, 224, 224).astype(np.float32))

# 执行模型
for _ in range(10):
    output = net(x)

# 结束Profiling并导出数据
profiler.analyse()

[1846] Start parsing profiling data in sync mode at: /home/developer/experiment/lab_10/data/profiler/1a116152ae73457393fc575388e8d685_1846_20260804024548820_ascend_ms
[2686] Parsing: [####################] 3/3 Done Elapsed: 4s                     4s                    


#### 5.2.3 使用MindStudio Insight可视化分析

MindStudio Insight工具用于将性能数据可视化展示，便于直观地分析性能瓶颈。

**安装MindStudio Insight**：

参考《MindStudio Insight工具用户指南》完成安装，确保工具版本与CANN版本兼容。

**导入性能数据**：

1. 启动MindStudio Insight工具
2. 选择“File → Import Profiling Data”
3. 选择上一节采集生成的PROF目录（或TensorBoard日志目录）
4. 工具将自动解析并展示性能数据

**查看Timeline视图**：

Timeline视图以时间轴形式展示各算子执行的时间分布，横轴为时间，纵轴为流（Stream），不同颜色代表不同类型的事件：

- 绿色区块：AI Core计算执行时间
- 蓝色区块：数据搬运（Memcpy）操作
- 红色区块：Host到Device的数据传输
- 灰色区块：任务调度等待时间

通过Timeline视图可以直观地观察到：
- 算子执行是否存在大量空隙（硬件利用率低）
- 数据传输是否与计算重叠
- 是否存在大量小算子导致的频繁调度开销

**查看算子耗时排名**：

在“Operator Statistics”面板中，可以看到各算子的耗时排名，找出计算热点。

### 5.3 任务三：性能瓶颈识别与优化实践

#### 5.3.1 常见性能瓶颈识别方法

基于采集的性能数据，可以从以下几个方面识别瓶颈：

**瓶颈类型一：Kernel Launch开销过大**

- **表现**：大量小算子（如Element-wise操作）导致频繁Kernel启动
- **诊断特征**：Timeline视图中出现大量短小的计算区块，op_statistic中算子调用次数远多于预期
- **根因**：Python层面的循环或条件分支导致算子无法融合，每个小算子都独立下发到NPU执行

**瓶颈类型二：内存带宽受限**

- **表现**：AI Core利用率低（<50%），但模型计算密度高
- **诊断特征**：L2缓存命中率 < 80%，或流水线利用率低
- **根因**：数据布局不合理导致缓存命中率下降，访存模式不连续

**瓶颈类型三：数据传输成为瓶颈**

- **表现**：Host-to-Device或Device-to-Host拷贝时间超过计算时间
- **诊断特征**：Timeline中出现长段数据传输标记，api_statistic中memcpy相关API耗时高
- **根因**：频繁的CPU与NPU之间的数据交互

**瓶颈类型四：AI CPU算子瓶颈**

- **表现**：部分算子运行在AI CPU上，执行效率远低于AI Core
- **诊断特征**：Timeline中出现AI CPU执行区块，op_summary中部分算子标注为AI CPU执行
- **根因**：数据类型不兼容（如int64类型算子）、算子不支持AI Core执行等

#### 5.3.2 优化策略实践

**优化策略一：算子融合**

算子融合是消除Kernel Launch开销最有效的手段之一。将多个相邻的小算子合并为一个Kernel执行，减少Host-Device的调度开销。

```python
# 优化前：多个独立的小算子，每个都需要独立下发
x = ops.relu(x)
x = ops.add(x, bias)
x = ops.mul(x, scale)

# 优化后：使用融合算子或图编译优化
# CANN的图编译器会自动进行算子融合，前提是使用图模式执行
import torchair as tng
config = CompilerConfig()
npu_backend = tng.get_npu_backend(compiler_config=config)
model = torch.compile(model, backend=npu_backend, fullgraph=True)
```

**优化策略二：数据布局调整**

昇腾NPU使用NC1HWC0亲和的5维数据布局格式，与常见的NCHW或NHWC不同。调整数据布局可以有效提升内存访问效率，提高L2缓存命中率。

```python
# 通过设置allow_internal_format允许CANN自动选择最优的数据布局
torch.npu.config.allow_internal_format = True
```

**优化策略三：AI CPU算子迁移**

通过调整数据类型，将原本在AI CPU上执行的算子迁移到AI Core上执行。以Equal算子为例：

```python
# 优化前：Equal算子数据类型为int64，在AI CPU上执行
equal_op = ops.Equal()
result = equal_op(x_int64, y_int64)

# 优化后：将数据类型改为FP16，迁移到AI Core上执行
x_fp16 = x_int64.to(torch.float16)
result_fp16 = equal_op(x_fp16, y_fp16.to(torch.float16))
```

#### 5.3.3 使用专家系统进行自动化分析

CANN提供了专家系统工具msadvisor，可以自动识别性能问题并给出优化建议。

```bash
# 配置环境变量
source /home/HwHiAiUser/Ascend/ascend-toolkit/set_env.sh

# 使用msadvisor进行AI CPU算子识别分析
# -d 参数指定profiling数据的路径（数据目录根路径）
# -c 参数配置值为model，表示对模型进行分析
msadvisor -d /path/to/profiling/data -c model
```

执行完成后，系统会以打屏形式输出分析结果，指出哪些算子在AI CPU上执行并给出具体的优化建议。

## 六、任务拓展

### 6.1 拓展实验一：比较图模式与Eager模式的性能差异

分别使用PyTorch Eager模式和图模式（torch.compile）执行同一模型，采集并对比两种模式下的性能数据：

```python
# Eager模式（默认模式）
# 直接运行模型即可，不需要特殊配置

# 图模式
import torchair as tng
from torchair.configs.compiler_config import CompilerConfig

config = CompilerConfig()
npu_backend = tng.get_npu_backend(compiler_config=config)
model = torch.compile(model, backend=npu_backend, fullgraph=True)
```

使用msprof分别采集两种模式下的性能数据，重点比较：
- 算子数量是否减少（图模式会自动进行算子融合）
- Kernel Launch开销是否降低
- 总体推理时延是否有改善

### 6.2 拓展实验二：多流并行优化

通过使用多流（Multi-Stream）技术，将数据传输与计算操作分配到不同的流上并行执行，隐藏数据拷贝的开销：

```python
# 创建多个流
stream1 = torch.npu.Stream()
stream2 = torch.npu.Stream()

# 流1：拷贝batch0并进行计算
with torch.npu.stream(stream1):
    data0 = data0.to("npu", non_blocking=True)
    output0 = model(data0)

# 流2：拷贝batch1并进行计算（与流1并行）
with torch.npu.stream(stream2):
    data1 = data1.to("npu", non_blocking=True)
    output1 = model(data1)

# 同步所有流
torch.npu.synchronize()
```

使用msprof采集性能数据，观察Timeline视图中数据传输与计算是否实现了并行重叠。

### 6.3 拓展实验三：模型量化对性能的影响

使用CANN的模型量化工具（如AMCT）对模型进行量化（FP16→INT8），对比量化前后模型的推理性能变化：

1. 使用AMCT工具对模型进行INT8量化
2. 采集量化前后模型的性能数据
3. 对比分析：推理时延变化、模型大小变化、精度损失情况

## 七、实验总结

本实验围绕CANN性能分析工具链，系统性地介绍了AI模型性能分析与优化的完整流程。通过本实验，同学们应当掌握以下核心能力：

**1. 性能数据采集能力**：熟练使用msprof命令行工具和框架Profiler接口（Ascend PyTorch Profiler / MindSpore Profiler）采集NPU上的性能数据，理解各采集参数的含义和适用场景。

**2. 性能数据分析能力**：能够解读op_summary.csv、op_statistic.csv、api_statistic.csv等核心输出文件，掌握使用MindStudio Insight进行Timeline视图分析和算子耗时排名的能力。

**3. 瓶颈定位能力**：能够识别Kernel Launch开销过大、内存带宽受限、数据传输瓶颈、AI CPU算子瓶颈等常见性能问题的诊断特征，准确找到性能瓶颈的根因。

**4. 性能优化能力**：掌握算子融合、数据布局调整、多流并行、算子迁移等常用优化策略，并能够验证优化效果。

性能优化是一个迭代的过程——从Profiling入手定位热点，分析访存、检查精度与算子选择，随后通过内存复用、亲和性绑定、批次与并行度调节等手段迭代优化。实验完成后，建议同学们总结自身的心得体会，梳理实验中遇到的问题及解决方法，形成完整的性能分析调优方法论。